# AsturCamín · Afinado del modelo de decisión Laya (Google Colab, 1× T4)

Versión para Colab del notebook de afinado. Los datos de AsturCamín van incrustados: no hace falta subir nada.

**Antes de ejecutar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar**.

Después: **Entorno de ejecución → Ejecutar todas**. Tarda alrededor de una hora; deja la pestaña abierta.
Al terminar, el navegador descarga `laya_asturcamin.zip` (si no, búscalo en el panel de archivos, a la izquierda).


## 1. Environment & Dual T4 GPU Check
Verify both T4 GPUs are detected.


In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 1, (
    f"No se detecta ninguna GPU.\n"
    "Activa la GPU: menú Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU T4."
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("GPU lista para entrenar.")


## 2. Install Dependencies


In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy accelerate tabulate
import laya, transformers, datasets, torch
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)


## 2b. Datos de AsturCamín

In [ ]:
# Datos de entrenamiento de AsturCamín (generados con modelo/exportar_finetune.py)
import base64, gzip, os
os.makedirs("/content/datos", exist_ok=True)
FICHEROS = {
    "finetune_train.jsonl": "...",
    "finetune_val.jsonl": "...",
}
for nombre, b64 in FICHEROS.items():
    with open(f"/content/datos/{nombre}", "wb") as fh:
        fh.write(gzip.decompress(base64.b64decode(b64)))
    n = sum(1 for _ in open(f"/content/datos/{nombre}", encoding="utf-8"))
    print(f"{nombre}: {n} avisos")


## 3. Modelo base y preprocesado

In [ ]:
import os, json, torch
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.train import items_from_rows

MODEL_ID = "convaiinnovations/laya"
print(f"Descargando {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID)

# Los avisos están en español: se entrena sobre el checkpoint multilingüe si tiene la estructura completa
multi = os.path.join(model_dir, "multilingual")
necesarios = ["tokenizer", "encoder", "rl_agent_config.json", "model.safetensors"]
if all(os.path.exists(os.path.join(multi, n)) for n in necesarios):
    model_dir = multi
    print("Base: checkpoint MULTILINGÜE")
else:
    print("AVISO: el checkpoint multilingüe no tiene la estructura esperada; se usa la base en inglés.")
    print("Contenido de multilingual/:", os.listdir(multi) if os.path.isdir(multi) else "no existe")
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

rows = []
with open("/content/datos/finetune_train.jsonl", encoding="utf-8") as fh:
    for linea in fh:
        row = json.loads(linea)
        rows.append({
            "state": json.loads(row["state"]),
            "questions": json.loads(row["questions"]),
            "gold": json.loads(row["gold"]),
        })

items, skipped = items_from_rows(tok, rows, cfg["max_len"], cfg["head_max_len"])
print(f"Preprocesadas {len(items)} preguntas de entrenamiento (descartadas {skipped}) de {len(rows)} avisos.")
torch.save(items, "/content/train_items.pt")


## 4. DDP Training Script (`train_ddp.py`)
We write the multi-GPU distributed RLCD training script using pure policy gradients with proper scoring rules and DDP gradient synchronization.


In [ ]:
%%writefile /content/train_ddp.py
"""2xT4 DDP training script for the typed-decisions fine-tune.

Training semantics -- item construction, encoding/collation, the RLCD objective,
the sigma schedule, the calibration holdout and the temperature fit -- come from
the canonical primitives in `laya.train` / `laya.common` / `laya.calibrate`.
What intentionally stays notebook-private is the distributed orchestration:
process-group setup, rank/local-rank wiring, the DDP wrapper, equal-rank
sharding and the effective-batch accounting.
"""
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file
from transformers import AutoTokenizer

from laya.common import build_model, collate_items
from laya.calibrate import fit_temperature_map
from laya.train import (
    calibration_records,
    encode_item,
    rlcd_loss,
    save_checkpoint,
    sigma_at,
    split_calibration,
)

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]

    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256
    max_len = cfg["max_len"]
    head_max_len = cfg["head_max_len"]

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))

    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)

    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)

    all_items = torch.load("/content/train_items.pt", weights_only=False)

    # Held out before sharding: `split_calibration` is seed-deterministic and
    # rank-independent, so every rank withholds the same items, and the shard
    # split happens after -- the calibration slice never reaches a training batch.
    train_items, calib_items = split_calibration(all_items, calib_max=400, calib_frac=0.1, seed=20260922)

    # Trim before sharding so every rank gets the same count (drops at most
    # world_size-1 items). Otherwise ranks can run different numbers of
    # micro-batches and DDP deadlocks on the unmatched all-reduce (#678).
    train_items = train_items[:len(train_items) // world_size * world_size]
    my_items = train_items[rank::world_size]

    EPOCHS = 6  # AsturCamín: 4 se quedaba corto para la severidad
    MICRO_BATCH = 8      # 8 sequences per forward pass per GPU
    GRAD_ACCUM = 8 // world_size  # lote efectivo de 64 secuencias con 1 o 2 GPU
    GROUP_SIZE = 4       # RLCD baseline samples
    LR_ENCODER = 2.5e-5  # Encoder adaptation rate
    LR_HEAD = 1.0e-4     # Head adaptation rate
    SIGMA_START = 0.4    # Exploration noise
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]

    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)

    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    if rank == 0:
        print(f"Starting 2xT4 DDP training: {len(train_items)} train items ({len(calib_items)} held out for calibration) | {len(my_items)} per rank | {EPOCHS} epochs")
    t0 = time.time()

    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0

        sigma = sigma_at(epoch, EPOCHS, SIGMA_START, SIGMA_END)
        steps_per_epoch = math.ceil(len(my_items) / MICRO_BATCH)

        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue

            encoded = [encode_item(tok, it, max_len, head_max_len) for it in chunk]
            batch = collate_items([encoded], tok.pad_token_id)

            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )

            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            target = batch["target"].to(device)

            # Canonical objective: RLCD policy gradient over noisy logits + full
            # soft cross-entropy guidance (`laya.train.rlcd_loss`).
            loss = rlcd_loss(logits, target, mask, batch["qtype"].to(device), sigma,
                             samples=GROUP_SIZE, w_sph=0.75, w_rps=1.0)

            window_start = (accum_step // GRAD_ACCUM) * GRAD_ACCUM
            window_size = min(GRAD_ACCUM, steps_per_epoch - window_start)
            scaled = loss / window_size
            scaler.scale(scaled).backward()
            accum_step += 1

            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item()
            n_batches += 1

            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item():.4f} | LR: {cur_lr:.2e}")

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()

        # Overwrite a single rolling checkpoint after each epoch so a crash,
        # OOM, or Kaggle session timeout doesn't lose all prior training.
        if rank == 0:
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            save_checkpoint(model, tok, dict(cfg, max_len=max_len, head_max_len=head_max_len), ckpt_dir)
            with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
                json.dump({
                    "epoch": epoch + 1,
                    "total_epochs": EPOCHS,
                    "avg_loss": epoch_loss / max(1, n_batches)
                }, f, indent=2)
            print(f"  Saved rolling checkpoint (epoch {epoch+1}/{EPOCHS}) to {ckpt_dir}")

    dist.barrier()

    # Post-training temperature calibration on rank 0
    if rank == 0:
        print("\nFitting post-training calibration temperatures...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        try:
            records = calibration_records(model, tok, calib_items, device, max_len, head_max_len, batch_size=16)
            fitted = fit_temperature_map(records)
            fitted_temps = fitted["temperature"]
            print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
            fitted_temps, fitted = None, {"temperature_by_options": {}}

        out_cfg = dict(cfg, max_len=max_len, head_max_len=head_max_len)
        out_cfg["fine_tuned"] = True
        out_cfg["model_name"] = "laya-asturcamin"
        if fitted_temps is not None:
            out_cfg["temperature"] = fitted_temps
            # This fit carries per-bucket overrides; inherited ones would shadow them.
            out_cfg.pop("temperature_by_options", None)
            if fitted["temperature_by_options"]:
                out_cfg["temperature_by_options"] = fitted["temperature_by_options"]
        save_checkpoint(model, tok, out_cfg, output_dir)
        print(f"Model successfully saved to {output_dir}!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()


## 5. Launch Multi-GPU Fine-Tuning with `torchrun`
Runs on both T4 GPUs in parallel (~4 to 6 minutes total).


In [ ]:
import torch
OUTPUT_DIR = "/content/laya_asturcamin"
MODEL_DIR = model_dir

cmd = f"torchrun --standalone --nproc_per_node={torch.cuda.device_count()} /content/train_ddp.py {MODEL_DIR} {OUTPUT_DIR}"
print("Entrenando:", cmd)
!{cmd}


## 6. Medida sobre la validación

In [ ]:
# Medida sobre la validación (199 avisos que el modelo no ha visto al entrenar)
import json, time
import laya

agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")
NIVELES = ["Baja", "Media", "Alta", "Crítica"]
aciertos = {"organismo": 0, "severidad": 0, "riesgo_vial": 0, "urgente": 0, "inspeccion": 0}
lat, n = [], 0
with open("/content/datos/finetune_val.jsonl", encoding="utf-8") as fh:
    for linea in fh:
        row = json.loads(linea)
        state, questions, gold = (json.loads(row[k]) for k in ("state", "questions", "gold"))
        t0 = time.perf_counter()
        r = agent_ft.predict(state, questions)["answers"]
        lat.append((time.perf_counter() - t0) * 1000)
        n += 1
        aciertos["organismo"] += r["organismo"]["choice"] == gold["organismo"]["label"]
        p = r["severidad"].get("probabilities", {})
        nivel = int(max(p, key=p.get)) if p else int(round(r["severidad"]["score"]))
        aciertos["severidad"] += nivel == gold["severidad"]["label"]
        for k in ("riesgo_vial", "urgente", "inspeccion"):
            aciertos[k] += (r[k]["noul"] >= 0.5) == (gold[k]["label"] == "true")

print(f"=== Modelo afinado · validación ({n} avisos) ===")
for k, v in aciertos.items():
    print(f"{k:12s}{v}/{n} ({v / n:.0%})")
print(f"Latencia media en GPU: {sum(lat) / n:.0f} ms")


## 7. Empaquetar el modelo

In [ ]:
!cd /content && zip -qr laya_asturcamin.zip laya_asturcamin && ls -lh laya_asturcamin.zip
from google.colab import files
files.download("/content/laya_asturcamin.zip")
